# Assortativity Analysis of Toxicity in User-Reply Networks

This notebook auto-discovers exported subreddit networks (`edges_<name>.csv`, `nodes_<name>.csv`), builds a **directed graph** per subreddit, and computes **Newman’s attribute assortativity** (_r_) on a **binary toxicity label** (`toxicity_rate > THRESHOLD`). It then runs a **label-shuffle permutation null** (centered at the null mean) to obtain **z-scores** and **two-sided permutation p-values**, saving per-subreddit results and a combined overview.

**Inputs:** `../outputs/networks/edges_<sub>.csv`, `nodes_<sub>.csv`  
**Outputs:** `../results/assortativity/assortativity_<sub>.csv`, plus `assortativity_overview.csv`  
**Key knobs:** `THRESHOLD` (toxic cutoff), `N_PERM` (permutations), `TREAT_NAN_AS` (NaN handling), `RANDOM_SEED`  
For each subreddit we get: `r_empirical`, null mean/std, `z`, permutation `p`, label counts, and 95% null quantiles.

### Setup: Imports and Configuration

In [41]:
import os, math, random
from typing import Dict, Any
import numpy as np
import pandas as pd
import networkx as nx

NETWORKS_DIR = "../user_reply_networks"
RESULTS_DIR  = "../results/assortativity"
 
THRESHOLD    = 0.10     # toxic user if toxicity_rate > THRESHOLD
N_PERM       = 1000
RANDOM_SEED  = 42
TREAT_NAN_AS = 0.0      # set to None to exclude NaNs

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Small utilities
Helper functions to create output folders and auto-discover available subreddits from `edges_*.csv`.

In [43]:
def ensure_dir(p): os.makedirs(p, exist_ok=True)

def discover_subreddits(base_dir: str):
    files = [f for f in os.listdir(base_dir) if f.startswith("edges_") and f.endswith(".csv")]
    # edges_<name>.csv -> <name>
    return sorted([f[len("edges_"):-len(".csv")] for f in files])

## Build a directed graph from edges/nodes
Loads `edges_<name>.csv` and `nodes_<name>.csv`, attaches toxicity attributes to nodes, and constructs a weighted DiGraph.

In [45]:
def load_graph(name: str, base_dir: str):
    epath = os.path.join(base_dir, f"edges_{name}.csv")
    npath = os.path.join(base_dir, f"nodes_{name}.csv")
    edges = pd.read_csv(epath)
    nodes = pd.read_csv(npath)

    G = nx.DiGraph()
    # Add nodes with attributes
    if "Id" not in nodes.columns:
        raise ValueError(f"'Id' column not found in nodes_{name}.csv")
    tox = nodes.set_index("Id")["toxicity_rate"] if "toxicity_rate" in nodes.columns else pd.Series(index=nodes["Id"], dtype=float)
    if TREAT_NAN_AS is not None:
        tox = tox.fillna(TREAT_NAN_AS)

    for u in nodes["Id"]:
        attrs = {}
        if u in tox.index and pd.notna(tox.loc[u]):
            r = float(tox.loc[u])
            attrs["tox_rate"] = r 
            attrs["tox_bin"]  = int(r > THRESHOLD)
        G.add_node(u, **attrs)

    # Add edges
    if "Weight" in edges.columns:
        G.add_weighted_edges_from(edges[["Source","Target","Weight"]].itertuples(index=False, name=None))
    else:
        G.add_edges_from(edges[["Source","Target"]].itertuples(index=False, name=None))

    return G

## Assortativity on binary toxicity
Computes Newman’s attribute assortativity coefficient _r_ on the binary `tox_bin` node label.

In [47]:
# Measures attribute homophily — the tendency for nodes to connect to others that share the same attribute value
# +1.0	Perfect assortativity — nodes only connect to others with the same attribute (e.g., toxic→toxic, non-toxic→non-toxic)
# 0.0	No correlation — connections are random with respect to that attribute
# –1.0	Perfect disassortativity — nodes always connect to others with a different attribute (e.g., toxic→non-toxic)
def assortativity_binary(G: nx.DiGraph, attr="tox_bin") -> float: 
    return nx.attribute_assortativity_coefficient(G, attr)

## Permutation null model and z/p
Shuffles toxicity labels across nodes to estimate a null distribution of _r_, returning mean, SD, z-score, two-sided p, and quantiles.

In [49]:
def perm_null_z_p(G: nx.DiGraph, r_emp: float, attr="tox_bin", n_perm=1000, seed=42) -> Dict[str, Any]:
    rng = np.random.default_rng(seed)
    nodes = [n for n, d in G.nodes(data=True) if attr in d and d[attr] is not None and not (isinstance(d[attr], float) and math.isnan(d[attr]))]
    if len(nodes) == 0 or G.number_of_edges() == 0:
        return {"mean_null": np.nan, "std_null": np.nan, "z": np.nan, "p_perm": 1.0, "n_perm": 0}

    labels = np.array([G.nodes[n][attr] for n in nodes], dtype=float)
    r_vals = []
    for _ in range(n_perm):
        rng.shuffle(labels)
        for n, lab in zip(nodes, labels):
            G.nodes[n][attr] = lab
        r = assortativity_binary(G, attr)
        if not np.isnan(r):
            r_vals.append(r)

    r_vals = np.array(r_vals)
    if len(r_vals) == 0:
        return {"mean_null": np.nan, "std_null": np.nan, "z": np.nan, "p_perm": 1.0, "n_perm": 0}

    mu = float(np.mean(r_vals))
    sd = float(np.std(r_vals, ddof=1)) if len(r_vals) > 1 else 0.0
    if sd == 0.0 or np.isnan(sd):
        z = np.nan
        p = 1.0
    else:
        z = (r_emp - mu) / sd
        # two-sided perm p centered at null mean:
        p = (float(np.sum(np.abs(r_vals - mu) >= abs(r_emp - mu))) + 1.0) / (len(r_vals) + 1.0)

    return {
        "mean_null": mu,
        "std_null": sd,
        "z": float(z),
        "p_perm": float(p),
        "n_perm": int(len(r_vals)),
        "q2.5": float(np.quantile(r_vals, 0.025)),
        "q50":  float(np.quantile(r_vals, 0.50)),
        "q97.5":float(np.quantile(r_vals, 0.975)),
    }

## Analyze a single subreddit
Builds the graph for one subreddit, computes empirical _r_, runs the permutation test, prints a summary, and writes a result C

In [51]:
def analyze_one(name: str, base_dir: str, out_dir: str) -> pd.DataFrame:
    G = load_graph(name, base_dir)
    lbls = [d["tox_bin"] for _, d in G.nodes(data=True) if "tox_bin" in d]
    n0 = int(np.sum(np.array(lbls) == 0))
    n1 = int(np.sum(np.array(lbls) == 1))
    r_emp = assortativity_binary(G, "tox_bin")

    res = {
        "subreddit": name,
        "n_nodes": G.number_of_nodes(),
        "n_edges": G.number_of_edges(), 
        "threshold": THRESHOLD,
        "n_tox0": n0,
        "n_tox1": n1,
        "r_empirical": float(r_emp) if not np.isnan(r_emp) else np.nan,
    }
    res.update(perm_null_z_p(G, r_emp, "tox_bin", N_PERM, RANDOM_SEED))

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"assortativity_{name}.csv")
    pd.DataFrame([res]).to_csv(out_path, index=False)
    print(f"[OK] {name}: r={res['r_empirical']:.4f} | z={res['z'] if not math.isnan(res['z']) else 'NaN'} | p={res['p_perm']:.4f} → {out_path}")
    return pd.DataFrame([res])

## Run analysis for all subreddits
Auto-discovers subreddits, iterates `analyze_one` across them, and saves a combined `assortativity_overview.csv`.

In [53]:
def run_all_assortativity():
    base_dir = NETWORKS_DIR
    out_dir  = RESULTS_DIR
    ensure_dir(out_dir)

    subs = discover_subreddits(base_dir)
    if not subs:
        raise FileNotFoundError(f"No edges_*.csv files found in {base_dir}")
    all_rows = []
    for name in subs:
        try:
            all_rows.append(analyze_one(name, base_dir, out_dir))
        except Exception as e:
            print(f"[WARN] Skipping {name}: {e}")

    if all_rows:
        overview = pd.concat(all_rows, ignore_index=True) 
        overview_path = os.path.join(out_dir, "assortativity_overview.csv")
        overview.to_csv(overview_path, index=False)
        print(f"\n[OK] Wrote combined overview → {overview_path}")

# Execute
run_all_assortativity()

[OK] AlbionOnline: r=0.0659 | z=3.418780439132849 | p=0.0020 → ../results/assortativity/assortativity_AlbionOnline.csv
[OK] DarkSouls: r=0.0687 | z=3.5431688009343465 | p=0.0030 → ../results/assortativity/assortativity_DarkSouls.csv
[OK] ElderScrollsOnline: r=0.0566 | z=5.962662161868989 | p=0.0010 → ../results/assortativity/assortativity_ElderScrollsOnline.csv
[OK] HollowKnight: r=0.0727 | z=5.482213689307967 | p=0.0010 → ../results/assortativity/assortativity_HollowKnight.csv
[OK] LiesofP: r=0.0447 | z=3.3007712140600916 | p=0.0020 → ../results/assortativity/assortativity_LiesofP.csv
[OK] Ninesols: r=0.0664 | z=3.6993872658885634 | p=0.0010 → ../results/assortativity/assortativity_Ninesols.csv
[OK] SkyChildrenoftheLight: r=0.2506 | z=4.038223043802521 | p=0.0030 → ../results/assortativity/assortativity_SkyChildrenoftheLight.csv
[OK] WoW: r=0.0376 | z=5.321171977314064 | p=0.0010 → ../results/assortativity/assortativity_WoW.csv

[OK] Wrote combined overview → ../results/assortativity/